In [11]:
# Import necessary libraries
import pandas as pd
import yfinance as yf
from datetime import datetime
from io import StringIO

print("🚀 Starting the automated data pipeline...")

🚀 Starting the automated data pipeline...


In [12]:
# Define time range
start_date = "2014-01-01"
end_date = datetime.today().strftime('%Y-%m-%d') # Automatically gets today's date

print(f"⏳ Downloading Brent Oil and USD/EGP rates up to {end_date}...")

# 1. Download Brent Crude Oil (BZ=F)
oil = yf.download("BZ=F", start=start_date, end=end_date, progress=False, auto_adjust=False)
if isinstance(oil.columns, pd.MultiIndex):
    oil = oil['Close']
else:
    oil = oil[['Close']]
oil.columns = ['Oil_Price']
oil = oil.apply(pd.to_numeric, errors='coerce').dropna()

# 2. Download USD to EGP rate (EGP=X)
usd = yf.download("EGP=X", start=start_date, end=end_date, progress=False, auto_adjust=False)
if isinstance(usd.columns, pd.MultiIndex):
    usd = usd['Close']
else:
    usd = usd[['Close']]
usd.columns = ['USD_Rate']
usd = usd.apply(pd.to_numeric, errors='coerce').dropna()

print("✅ Global market data downloaded successfully.")

⏳ Downloading Brent Oil and USD/EGP rates up to 2026-09-10...
✅ Global market data downloaded successfully.


In [13]:
# Merge oil and USD datasets on date
market_df = pd.merge(oil, usd, left_index=True, right_index=True, how='inner')
market_df.reset_index(inplace=True)
market_df.rename(columns={'index': 'Date'}, inplace=True)
market_df['Date'] = pd.to_datetime(market_df['Date'])
market_df = market_df.sort_values('Date')

# Calculate 3-month (90-day) moving averages to simulate the pricing committee methodology
market_df.set_index('Date', inplace=True)
market_df['Oil_3M_Avg'] = market_df['Oil_Price'].rolling(window=90).mean()
market_df['USD_3M_Avg'] = market_df['USD_Rate'].rolling(window=90).mean()
market_df.reset_index(inplace=True)

# Save market history file
market_df.to_csv('market_factors_history.csv', index=False)
print("✅ Market factors history saved successfully: market_factors_history.csv")

✅ Market factors history saved successfully: market_factors_history.csv


In [14]:
# Read local fuel prices dataset
print("⚙️ Loading local fuel prices and merging with market factors...")
df_fuel = pd.read_csv('raw data 1.csv')
df_fuel['Date'] = pd.to_datetime(df_fuel['Date']).astype('datetime64[ns]')
df_fuel = df_fuel.sort_values('Date')

market_df['Date'] = pd.to_datetime(market_df['Date']).astype('datetime64[ns]')

# Use merge_asof to align local pricing events with the latest available macroeconomic averages
final_df = pd.merge_asof(
    df_fuel,
    market_df,
    on='Date',
    direction='backward'
)

# Clean missing rows
final_df = final_df.dropna(subset=['Oil_3M_Avg', 'USD_3M_Avg'])
print("✅ Data merge completed successfully.")

⚙️ Loading local fuel prices and merging with market factors...
✅ Data merge completed successfully.


In [15]:
# Save the final training dataset for Machine Learning models
output_file = 'final_training_dataset.csv'
final_df.to_csv(output_file, index=False)

print("=" * 50)
print(f"🎉 Pipeline completed! Final dataset saved as: {output_file}")
print(f"📊 Total training records available: {len(final_df)}")
print("=" * 50)

# Display sample of the updated dataset
print(final_df[['Date', 'Fuel_Type', 'Price', 'Oil_3M_Avg', 'USD_3M_Avg']].tail())

🎉 Pipeline completed! Final dataset saved as: final_training_dataset.csv
📊 Total training records available: 88
         Date          Fuel_Type  Price  Oil_3M_Avg  USD_3M_Avg
83 2025-10-17  Natural Gas (CNG)  10.00   67.957778   48.683259
84 2025-10-17     Diesel (Solar)  17.50   67.957778   48.683259
85 2025-10-17          Octane 95  21.00   67.957778   48.683259
86 2025-10-17          Octane 92  19.25   67.957778   48.683259
87 2025-10-17          Octane 80  17.75   67.957778   48.683259
